# MedExplain AI — Patient Medical Report & Lab Result Assistant

MedExplain AI is a conversational patient education assistant that explains medical terminology, laboratory results, and medical report information in simple language.

The goal of this prototype is to apply the conversational AI concepts from Day 3 to the healthcare domain.

The assistant uses:

- A healthcare-specific system prompt
- Conversation history
- Streaming responses
- Conditional prompting
- A Gradio chat interface

This project is intended for educational purposes and does not provide medical diagnosis or treatment.

In [1]:
# Step1: importing libraries

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Step 2:Load API Key

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("OpenAI API Key is ready")
else:
    print("OpenAI API Key not set")

OpenAI API Key is ready


In [3]:
# Step 3: Initialize OpenAI

openai = OpenAI()

MODEL = "gpt-4.1-mini"

## MedExplain System Prompt

The system prompt gives the LLM context about its role.

For MedExplain AI, the assistant should explain medical information in simple language while clearly avoiding diagnosis or treatment recommendations.

In [4]:
# Step 4: MedExplain System Prompt
system_message = """
You are MedExplain AI, a patient education assistant.

Your role is to help patients understand medical terminology,
laboratory results, and medical report information in simple language.

Use a calm, clear, and supportive tone.

When explaining medical information:

1. Explain what the medical term or test means.
2. Explain the result in simple language.
3. Explain whether the value appears low, normal, or high only when
   sufficient reference information is available.
4. Explain possible general reasons for abnormal results without
   diagnosing the patient.
5. Suggest useful questions the patient may want to ask their
   healthcare provider.

Important rules:

- Do not diagnose medical conditions.
- Do not prescribe medications or treatments.
- Do not claim that the user definitely has a disease.
- Clearly state when more information is needed.
- Encourage the user to discuss concerning results with a qualified
  healthcare professional.
- If the user describes symptoms that may represent an emergency,
  recommend seeking urgent medical care.

Your purpose is education and explanation, not medical diagnosis.
"""

In [5]:
# Step 5: MedExplain Chat Function

def chat(message, history):
    
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [6]:
# Step 6: Gradio Chat Interface
gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


Examples: tachycardia, inflamanation, hypertension

# Phase 2 of the Project:

## Conditional Medical Context

We can make MedExplain more useful by changing the system prompt depending on the type of question asked by the user.

For example:
- Lab result questions should explain the value and reference range
- Medication questions should explain general medication information
- Emergency symptoms should trigger stronger safety guidance

In [26]:
def chat(message, history):

    system_prompt = system_message

    if any(word in message.lower() for word in [
        "hba1c", 
        "cholesterol", 
        "ldl", 
        "hdl", 
        "glucose", 
        "hemoglobin", 
        "wbc", 
        "rbc", 
        "tsh"]):

        system_prompt += """

The user appears to be asking about a laboratory result.

When explaining the result:

- Explain what the test measures
- Explain the value in simple language
- Discuss typical reference ranges only when appropriate
- Explain possible general reasons for high or low values
- Do not diagnose a condition
- Recommend discussing abnormal or concerning results with a healthcare professional
"""

    elif any(word in message.lower() for word in ["medication", "medicine", "drug", "tablet", "dose"]):

        system_prompt += """
The user appears to be asking about medication.

Provide general educational information about the medication.

Explain:

- What the medication is commonly used for
- How it generally works
- Common side effects
- Important general precautions

Do not prescribe medication.
Do not recommend changing or stopping a dose.
Encourage the user to speak with a doctor or pharmacist for personal medical advice.
"""

    elif any(word in message.lower() for word in [
        "chest pain",
        "difficulty breathing",
        "shortness of breath",
        "unconscious",
        "severe bleeding",
        "stroke"
    ]):

        system_prompt += """
The user may be describing a potentially serious medical emergency.

Prioritize safety.

Clearly recommend seeking immediate medical attention or contacting emergency services.

Do not attempt to diagnose the condition.
Do not provide lengthy explanations before giving the urgent safety recommendation.
"""

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [8]:
gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()


* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


# Phase 3 of the Project: Medical Report Upload

- It support the PDF and TXT reports. 
- We will extract their text and let MedExplain expalin the report

In [9]:
# Step 1: Import Libraries

from pypdf import PdfReader

print("pypdf is ready")

pypdf is ready


## Step 2: Medical Report Upload

- MedExplain can also read an uploaded medical report.

- The report text will be extracted and provided to the LLM as additional context so that the assistant can explain the information in simple language.

In [27]:
def read_medical_report(file):
    if file is None:
        return ""

    if file.name.lower().endswith(".pdf"):
        reader = PdfReader(file.name)
        report_text = ""

        for page in reader.pages:
            text = page.extract_text()

            if text:
                report_text += text + "\n"

        return report_text

    elif file.name.lower().endswith(".txt"):
        with open(file.name, "r", encoding="utf-8") as f:
            return f.read()

    return ""

This above functions is:
- Takes input as PDF/TXT
- Read the Medical Report
- Convert it to plain text

## Step 3: Give Report to MedExplain

In [28]:
def analyze_report(file):

    report_text = read_medical_report(file)

    if not report_text:
        return "Please upload a PDF or TXT medical report."

    report_prompt = f"""
The following text was extracted from a medical report:
{report_text}
Explain this report to the patient in simple language.

Organize your response into:

## Report Summary

## Important Results

## What the Results Mean

## Results to Discuss With Your Healthcare Provider

## Questions You May Want to Ask Your Healthcare Provider

Do not diagnose the patient or prescribe treatment.
Clearly mention when information is missing or uncertain.
"""

    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": report_prompt}
        ]
    )

    return response.choices[0].message.content

## Step 4: Gradio Chat Interface

In [29]:
report_input = gr.File(label="Upload Medical Report",file_types=[".pdf", ".txt"])


report_output = gr.Markdown(label="Medical Report Explanation")

report_view = gr.Interface(
    fn=analyze_report,
    title="MedExplain AI - Medical Report Analyzer",
    description="Upload a medical report and receive a simple patient-friendly explanation.",
    inputs=[report_input],
    outputs=[report_output],
    flagging_mode="never"
)

In [30]:
report_view.launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\si

# Conclusion

MedExplain AI was developed in three phases.

### Phase 1 - Medical Chat Assistant
The first phase created a conversational healthcare assistant that explains medical terms and information in simple, patient-friendly language using a healthcare-specific system prompt.

### Phase 2 - Context-Aware Medical Assistant
The second phase improved the assistant by adding conversation history, streaming responses, and conditional prompting for lab results, medications, and potentially urgent symptoms.

### Phase 3 - Medical Report Analyzer
The third phase added medical report upload functionality, allowing users to upload PDF or TXT reports and receive a simple explanation of important results and information to discuss with a healthcare professional.

Overall, MedExplain AI demonstrates how LLMs can be applied to healthcare to make complex medical information easier for patients to understand while keeping the assistant focused on education rather than diagnosis or treatment.